In [ ]:
import os
import shutil
import logging
import tempfile
import pythoncom
import win32com.client
import openpyxl
import re
import time

from tkinter import Tk, Button, filedialog, messagebox
from typing import List, Optional, Tuple
from pypdf import PdfWriter


# ----------------------------------------------------------------------

def get_script_dir() -> str:
    return os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()


SCRIPT_DIR = get_script_dir()

PDF_OUTPUT_DIR = os.path.join(SCRIPT_DIR, "PDFs_Generados")
PDF_FLAT_DIR = os.path.join(SCRIPT_DIR, "Lista_PDFs")
CAL_DIR = os.path.join(SCRIPT_DIR, "Calibraciones")


KNOWN_BRANDS = {
    'FLUKE', 'FLK', 'B2', 'OMICRON', 'MEGABRAS',
    'AEMC', 'KEYSIGHT', 'AGILENT', 'MEGGER',
    'EXTECH', 'UNI-T', 'GWINSTEK', 'RIGOL',
    'TEKTRONIX', 'HIOKI', 'KAISE',
    'CHAUVINARNOUX', 'CA', 'KYORITSU',
    'AMPROBE', 'IDEAL', 'GREENLEE',
    'MARTINDALE', 'SEAWARD'
}


# ----------------------------------------------------------------------

def normalize(s: str) -> str:

    if not s:
        return ""

    s = str(s).strip().upper()

    return re.sub(r'[\s\.\-_/\\:]+', '', s)


# ----------------------------------------------------------------------

def normalize_date(date_str: str) -> str:
    """
    Convierte:
    18/02/2026
    ->
    18022026
    """

    if not date_str:
        return ""

    txt = str(date_str).strip()

    m = re.search(
        r'(\d{1,2})[\/\-](\d{1,2})[\/\-](\d{4})',
        txt
    )

    if not m:
        return ""

    dd = m.group(1).zfill(2)
    mm = m.group(2).zfill(2)
    yyyy = m.group(3)

    return f"{dd}{mm}{yyyy}"


# ----------------------------------------------------------------------

class ExcelToPdfConverter:

    def __init__(self):

        self.root = Tk()

        self.root.title("Excel to PDF - Protocolos L3")
        self.root.geometry("380x110")
        self.root.resizable(False, False)

        btn = Button(
            self.root,
            text="Cargar Excel(s) → PDF",
            font=("Segoe UI", 11, "bold"),
            bg="#1f6aa5",
            fg="white",
            command=self.run
        )

        btn.pack(expand=True, fill="both", padx=25, pady=25)

    # ------------------------------------------------------------------

    @staticmethod
    def kill_excel_processes() -> None:

        import psutil

        for proc in psutil.process_iter(['name']):

            if proc.info['name'] and proc.info['name'].lower() == 'excel.exe':

                try:
                    proc.terminate()
                except:
                    pass

    # ------------------------------------------------------------------

    @staticmethod
    def get_smart_used_range(filepath: str) -> Optional[Tuple[int, int, int, int]]:

        try:

            wb = openpyxl.load_workbook(
                filepath,
                read_only=True,
                data_only=True
            )

            ws = wb.active

            if ws.max_row < 1 or ws.max_column < 1:

                wb.close()

                return (1, 50, 1, 20)

            col_has_data = [False] * (ws.max_column + 20)

            real_max_row = 1
            observaciones_row = None

            for row_cells in ws.iter_rows(min_row=1, max_row=ws.max_row):

                for cell in row_cells:

                    if (
                        hasattr(cell, 'value')
                        and cell.value not in (None, "", " ", "\n", "\t")
                    ):

                        if hasattr(cell, 'column'):
                            col_has_data[cell.column] = True

                        real_max_row = max(real_max_row, cell.row)

                        if (
                            isinstance(cell.value, str)
                            and "OBSERVACION" in str(cell.value).upper()
                        ):

                            observaciones_row = max(
                                observaciones_row or 0,
                                cell.row
                            )

            max_col = 0
            empty_streak = 0

            for col in range(1, len(col_has_data)):

                if col_has_data[col]:

                    max_col = col
                    empty_streak = 0

                else:

                    empty_streak += 1

                    if empty_streak >= 4:
                        break

            if max_col == 0:
                max_col = 1

            final_max_row = real_max_row

            if observaciones_row:
                final_max_row = max(
                    final_max_row,
                    observaciones_row + 10
                )

            min_row = 1
            max_row = max(final_max_row, real_max_row + 4)
            min_col = 1

            logging.info(
                f"Rango calculado → "
                f"Filas {min_row}-{max_row} | "
                f"Columnas {min_col}-{max_col}"
            )

            wb.close()

            return (min_row, max_row, min_col, max_col)

        except Exception as e:

            logging.error(
                f"Error analizando rango {filepath}: {e}",
                exc_info=True
            )

            return None

    # ------------------------------------------------------------------

    def append_calibration_pdf(
        self,
        main_pdf_path: str,
        protocolo: str,
        ws
    ) -> str:

        if (
            not os.path.exists(main_pdf_path)
            or not os.path.exists(CAL_DIR)
        ):
            return main_pdf_path

        # =========================================================
        # LEER PDFs
        # =========================================================

        try:

            pdf_files = [
                f for f in os.listdir(CAL_DIR)
                if f.lower().endswith(".pdf")
            ]

            pdf_info = []

            for fname in pdf_files:

                name_no_ext = os.path.splitext(fname)[0]

                pdf_info.append({
                    "path": os.path.join(CAL_DIR, fname),
                    "name": name_no_ext,
                    "norm": normalize(name_no_ext)
                })

        except Exception as e:

            logging.error(f"Error leyendo calibraciones: {e}")

            return main_pdf_path

        if not pdf_info:

            logging.warning("No hay PDFs en carpeta Calibraciones")

            return main_pdf_path

        logging.info(
            f"PDFs calibración encontrados: "
            f"{[p['name'] for p in pdf_info]}"
        )

        # =========================================================
        # EXTRAER EQUIPOS DESDE EL EXCEL
        # =========================================================

        equipos = []

        current = {
            "brand": "",
            "model": "",
            "serial": "",
            "cal_date": ""
        }

        for row in ws.iter_rows(min_row=1, max_row=ws.max_row):

            values = []

            for c in row:

                try:
                    values.append(str(c.value).strip() if c.value else "")
                except:
                    values.append("")

            row_text = " | ".join(values).upper()

            # -----------------------------------------------------
            # BRAND
            # -----------------------------------------------------

            if "MARCA" in row_text or "BRAND" in row_text:

                for v in values:

                    nv = normalize(v)

                    for b in KNOWN_BRANDS:

                        if normalize(b) == nv:

                            current["brand"] = normalize(b)

            # -----------------------------------------------------
            # MODEL
            # -----------------------------------------------------

            if "MODELO" in row_text or "MODEL" in row_text:

                for v in values:

                    nv = normalize(v)

                    if (
                        re.search(r'[A-Z0-9]{3,}', nv)
                        and 2 <= len(nv) <= 30
                    ):

                        if nv not in (
                            "MODELO",
                            "MODEL"
                        ):

                            current["model"] = nv

            # -----------------------------------------------------
            # SERIAL
            # -----------------------------------------------------

            if (
                "SERIE" in row_text
                or "SERIAL" in row_text
            ):

                for v in values:

                    nv = normalize(v)

                    if (
                        len(nv) >= 5
                        and re.search(r'[A-Z0-9]{5,}', nv)
                    ):

                        current["serial"] = nv

            # -----------------------------------------------------
            # CALIBRATION DATE
            # -----------------------------------------------------

            if (
                "CALIBRACION" in row_text
                or "CALIBRATION" in row_text
            ):

                for v in values:

                    if not v:
                        continue

                    nd = normalize_date(v)

                    if nd:

                        current["cal_date"] = nd

                        logging.info(
                            f"Fecha calibración detectada: {nd}"
                        )

            # -----------------------------------------------------
            # GUARDAR EQUIPO
            # -----------------------------------------------------

            score_fields = sum([
                1 if current["brand"] else 0,
                1 if current["model"] else 0,
                1 if current["serial"] else 0,
                1 if current["cal_date"] else 0
            ])

            if score_fields >= 2:

                equipos.append(current.copy())

                logging.info(f"Equipo detectado: {current}")

                current = {
                    "brand": "",
                    "model": "",
                    "serial": "",
                    "cal_date": ""
                }

        if not equipos:

            logging.warning(
                "No se detectaron equipos dentro del Excel"
            )

            return main_pdf_path

        # =========================================================
        # MATCH PDFs
        # =========================================================

        matched_paths = []

        for equipo in equipos:

            best_score = 0
            best_pdf = None

            for pdf in pdf_info:

                score = 0

                pdf_norm = pdf["norm"]

                # BRAND
                if equipo["brand"]:

                    if equipo["brand"] in pdf_norm:
                        score += 50

                # MODEL
                if equipo["model"]:

                    if (
                        equipo["model"] in pdf_norm
                        or pdf_norm in equipo["model"]
                    ):
                        score += 35

                # SERIAL
                if equipo["serial"]:

                    if equipo["serial"] in pdf_norm:
                        score += 80

                # CAL DATE
                if equipo["cal_date"]:

                    if equipo["cal_date"] in pdf_norm:
                        score += 120

                logging.info(
                    f"[MATCH] "
                    f"{equipo} "
                    f"vs "
                    f"{pdf['name']} "
                    f"→ SCORE {score}"
                )

                if score > best_score:

                    best_score = score
                    best_pdf = pdf

            if best_pdf and best_score >= 50:

                if best_pdf["path"] not in matched_paths:

                    matched_paths.append(best_pdf["path"])

                    logging.info(
                        f"PDF seleccionado: "
                        f"{best_pdf['name']} "
                        f"(score {best_score})"
                    )

        if not matched_paths:

            logging.warning("No hubo matches de calibración")

            return main_pdf_path

        # =========================================================
        # MERGE PDFs
        # =========================================================

        try:

            writer = PdfWriter()

            with open(main_pdf_path, "rb") as f:
                writer.append(f)

            for pdf_path in matched_paths:

                logging.info(
                    f"Adjuntando calibración: {pdf_path}"
                )

                with open(pdf_path, "rb") as f:
                    writer.append(f)

            tmp_output = main_pdf_path.replace(
                ".pdf",
                "_temp.pdf"
            )

            with open(tmp_output, "wb") as out:
                writer.write(out)

            os.replace(tmp_output, main_pdf_path)

            logging.info("Calibraciones adjuntadas OK")

        except Exception as e:

            logging.error(
                f"Error mergeando calibraciones: {e}"
            )

        return main_pdf_path

    # ------------------------------------------------------------------

    def convert_file(self, src_path: str, excel_app) -> bool:

        try:

            base_name = os.path.splitext(
                os.path.basename(src_path)
            )[0]

            with tempfile.TemporaryDirectory() as tmp:

                tmp_xlsx = os.path.join(
                    tmp,
                    os.path.basename(src_path)
                )

                time.sleep(0.5)

                shutil.copy2(src_path, tmp_xlsx)

                time.sleep(0.8)

                wb_openpyxl = openpyxl.load_workbook(
                    tmp_xlsx,
                    read_only=True,
                    data_only=True
                )

                ws_openpyxl = wb_openpyxl.active

                smart_range = self.get_smart_used_range(tmp_xlsx)

                if not smart_range:

                    min_row = 1
                    max_row = min(ws_openpyxl.max_row or 500, 500)
                    min_col = 1
                    max_col = min(ws_openpyxl.max_column or 40, 40)

                else:

                    min_row, max_row, min_col, max_col = smart_range

                logging.info(
                    f"Área impresión: "
                    f"Filas {min_row}-{max_row}, "
                    f"Cols {min_col}-{max_col}"
                )

                wb = excel_app.Workbooks.Open(
                    os.path.abspath(tmp_xlsx),
                    ReadOnly=True,
                    UpdateLinks=0,
                    IgnoreReadOnlyRecommended=True
                )

                ws = wb.Worksheets(1)

                print_area = ws.Range(
                    ws.Cells(min_row, min_col),
                    ws.Cells(max_row, max_col)
                )

                ws.PageSetup.PrintArea = print_area.Address

                try:

                    ws.PageSetup.Zoom = False
                    ws.PageSetup.FitToPagesWide = 1
                    ws.PageSetup.FitToPagesTall = False
                    ws.PageSetup.Orientation = 2

                    try:
                        ws.PageSetup.PaperSize = 1
                    except:
                        pass

                    margin = excel_app.InchesToPoints(0.25)

                    ws.PageSetup.LeftMargin = margin
                    ws.PageSetup.RightMargin = margin
                    ws.PageSetup.TopMargin = margin
                    ws.PageSetup.BottomMargin = margin
                    ws.PageSetup.HeaderMargin = margin
                    ws.PageSetup.FooterMargin = margin
                    ws.PageSetup.CenterHorizontally = True

                except Exception as e:

                    logging.warning(f"PageSetup falló: {e}")

                tmp_pdf = os.path.join(
                    tmp,
                    f"{base_name}.pdf"
                )

                wb.ExportAsFixedFormat(0, tmp_pdf)

                wb.Close(False)

                del ws
                del wb

                import gc
                gc.collect()

                time.sleep(1.5)

                if not os.path.exists(tmp_pdf):

                    wb_openpyxl.close()

                    return False

                asset_dir = os.path.join(
                    PDF_OUTPUT_DIR,
                    base_name
                )

                os.makedirs(asset_dir, exist_ok=True)

                final_pdf = os.path.join(
                    asset_dir,
                    f"{base_name}.pdf"
                )

                shutil.move(tmp_pdf, final_pdf)

                self.append_calibration_pdf(
                    final_pdf,
                    base_name,
                    ws_openpyxl
                )

                wb_openpyxl.close()

                flat_path = os.path.join(
                    PDF_FLAT_DIR,
                    f"{base_name}.pdf"
                )

                i = 1

                while os.path.exists(flat_path):

                    flat_path = os.path.join(
                        PDF_FLAT_DIR,
                        f"{base_name}_{i}.pdf"
                    )

                    i += 1

                shutil.copy2(final_pdf, flat_path)

            return True

        except Exception as e:

            logging.error(
                f"Fallo convirtiendo {src_path}: {e}",
                exc_info=True
            )

            return False

    # ------------------------------------------------------------------

    def run(self):

        files: List[str] = filedialog.askopenfilenames(
            title="Seleccionar protocolos Excel (.xlsx)",
            filetypes=[("Excel files", "*.xlsx *.xls")]
        )

        if not files:
            return

        os.makedirs(PDF_OUTPUT_DIR, exist_ok=True)
        os.makedirs(PDF_FLAT_DIR, exist_ok=True)

        total = len(files)
        processed = 0

        self.kill_excel_processes()

        pythoncom.CoInitialize()

        excel = None

        try:

            excel = win32com.client.Dispatch(
                "Excel.Application"
            )

            excel.Visible = False
            excel.DisplayAlerts = False
            excel.ScreenUpdating = False

            for fp in files:

                logging.info(f"Procesando: {fp}")

                if self.convert_file(fp, excel):
                    processed += 1

            messagebox.showinfo(
                "Conversión Completada",
                f"Éxito: {processed} de {total} archivos convertidos\n\n"
                f"PDFs organizados → {PDF_OUTPUT_DIR}\n"
                f"Lista plana → {PDF_FLAT_DIR}"
            )

        except Exception as e:

            messagebox.showerror(
                "Error Crítico",
                f"Error general: {e}"
            )

        finally:

            if excel:

                try:
                    excel.Quit()
                except:
                    self.kill_excel_processes()

            pythoncom.CoUninitialize()

    # ------------------------------------------------------------------

    def start(self):

        self.root.mainloop()


# ----------------------------------------------------------------------

if __name__ == "__main__":

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[logging.StreamHandler()]
    )

    app = ExcelToPdfConverter()

    app.start()

2026-07-06 10:10:58,081 | INFO | Procesando: C:/Users/JoaquinAzpiroz/Downloads/LV temporal Cable ATS B1-2-b-21 (1).xlsx
2026-07-06 10:10:59,516 | INFO | Rango calculado → Filas 1-108 | Columnas 1-9
2026-07-06 10:10:59,518 | INFO | Área impresión: Filas 1-108, Cols 1-9
2026-07-06 10:11:14,238 | INFO | PDFs calibración encontrados: ['AEMC 6240 222976YLH 24102025', 'AEMC 6505 150165UKDV 18022026', 'AEMC 6550 111470YLDV 13052026', 'AEMC 6550 111470YLDV 22102025', 'AEMC 8510 DTR 113070ZADV 23102025', 'B2 HVA68-2 GH5228.22I001 01102025', 'B2 HVA68-2 GH5228.22I001 14052026', 'B2 PDTD60-2 GH5728.22A017 14052026', 'Baur Viola TD 1446418001 08062026', 'Eurotech 5530 - 10-120 Nm  250600157 03022026', 'Eurotech 5530 - 5-25 Nm 250600060 02022026', 'Eurotech 5530 - 60-330 Nm 250600292 02022026', 'Eurotech 5530 250600060 02022026', 'Eurotech 5530 250600157 03022026', 'Eurotech 5530 250600292 03022026', 'FLUKE 115 45160680SV 06042026', 'FLUKE 377 63700002 24102025', 'FLUKE T6-1000 59220035 13042026', 